In [3]:
# QUESTION 1.1

import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLP resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_json('research_abstracts.json')

# Display dataset information
print("Dataset columns:", df.columns.tolist())
print(df.head())
print(df.info())
print("\nMissing values:\n", df.isnull().sum())

# Identify abstract column automatically
abstract_col = next(
    (col for col in df.columns if 'abstract' in col.lower()),
    None
)

if abstract_col is None:
    raise ValueError("No abstract column found. Check df.columns.")

# Initialise preprocessing tools
stop_words = set(stopwords.words('english')) - {'no', 'nor', 'not'}
lemmatizer = WordNetLemmatizer()

# Clean text
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'<[^>]+>|http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(words)

# Apply preprocessing
df = df.dropna(subset=[abstract_col]).copy()
df['cleaned_abstract'] = df[abstract_col].apply(clean_text)

# Remove empty and duplicate records
df = df[df['cleaned_abstract'] != ''].copy()
df = df.drop_duplicates(subset=['cleaned_abstract'])

# Display cleaned results
print("\nCleaned abstracts:")
print(df[['cleaned_abstract']].head())

print("\nFinal dataset shape:", df.shape)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mosa.Mefane\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mosa.Mefane\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Dataset columns: ['DOI', 'Abstract', 'Label']
                          DOI  \
0        10.1155/2021/1071145   
1  10.1007/s43546-022-00384-2   
2            10.1002/for.2667   
3  10.1007/s42973-021-00091-x   
4   10.1007/s40258-012-0003-z   

                                            Abstract      Label  
0  Forward-looking forecasting of the inflation r...         ai  
1  Our study examined the disaggregation of infla...         ai  
2  This paper develops a dynamic factor model tha...  classical  
3  In this paper, I use a simple SIR Macro model ...  classical  
4  BackgroundGame theory is useful for identifyin...  classical  
<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1137
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DOI       1134 non-null   object
 1   Abstract  1138 non-null   object
 2   Label     1138 non-null   object
dtypes: object(3)
memory usage: 35.6+ KB
None

Missing values:
 DOI 

In [4]:
#1.2

# Import required libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Identify relevant dataset columns
print("Dataset columns:", df.columns.tolist())

# Select input feature and target variable
X_text = df['cleaned_abstract'].fillna('')
y = df['Label']

# Apply TF-IDF feature extraction
tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(X_text)

# Encode categorical target variable
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Display results
print("\nSelected Input Feature: cleaned_abstract")
print("Target Variable: Label")
print("Excluded Column: DOI")

print("\nTF-IDF Matrix Shape:", X.shape)
print("Target Variable Shape:", y_encoded.shape)

print("\nTop 20 Extracted Features:")
print(tfidf.get_feature_names_out()[:20])

print("\nResearch Categories:")
print(encoder.classes_)


Dataset columns: ['DOI', 'Abstract', 'Label', 'cleaned_abstract']

Selected Input Feature: cleaned_abstract
Target Variable: Label
Excluded Column: DOI

TF-IDF Matrix Shape: (1118, 1000)
Target Variable Shape: (1118,)

Top 20 Extracted Features:
['ability' 'able' 'absolute' 'abstract' 'according' 'account' 'accuracy'
 'accurate' 'achieve' 'activity' 'actual' 'adaptive' 'addition'
 'additional' 'address' 'adjusted' 'adjustment' 'advanced' 'advantage'
 'affect']

Research Categories:
['ai' 'classical']


In [5]:
#1.3

#Import required libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#Select input and target variables
X = df['cleaned_abstract'].fillna('')
y = df['Label']

#Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

#Create baseline classification model
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('classifier', MultinomialNB())
])

#Train the model
model.fit(X_train, y_train)

#Predict training and testing data
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

#Evaluate training performance
print("TRAINING PERFORMANCE")
print("Training Accuracy:",
      round(accuracy_score(y_train, train_pred) * 100, 2), "%")

#Evaluate testing performance
print("\nTESTING PERFORMANCE")
print("Testing Accuracy:",
      round(accuracy_score(y_test, test_pred) * 100, 2), "%")

#Generate classification report
print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, test_pred, zero_division=0))

#Display confusion matrix
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, test_pred))

#Perform 5-fold cross-validation
cv_scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring='accuracy'
)

print("\nCROSS-VALIDATION PERFORMANCE")
print("Cross-validation scores:", cv_scores)
print("Average accuracy:",
      round(cv_scores.mean() * 100, 2), "%")

#Compare training and testing performance
print("\nGENERALIZATION PERFORMANCE")
print("Accuracy gap:",
      round((accuracy_score(y_train, train_pred) -
             accuracy_score(y_test, test_pred)) * 100, 2), "%")


TRAINING PERFORMANCE
Training Accuracy: 91.83 %

TESTING PERFORMANCE
Testing Accuracy: 89.29 %

CLASSIFICATION REPORT
              precision    recall  f1-score   support

          ai       0.96      0.51      0.67        47
   classical       0.88      0.99      0.94       177

    accuracy                           0.89       224
   macro avg       0.92      0.75      0.80       224
weighted avg       0.90      0.89      0.88       224


CONFUSION MATRIX
[[ 24  23]
 [  1 176]]

CROSS-VALIDATION PERFORMANCE
Cross-validation scores: [0.89385475 0.86592179 0.88826816 0.8603352  0.90449438]
Average accuracy: 88.26 %

GENERALIZATION PERFORMANCE
Accuracy gap: 2.55 %


In [6]:
#Question 2
#2.1

# QUESTION 2.1: TRAIN FOUR MACHINE LEARNING MODELS

# Import required libraries

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Select input and target variables
X = df['cleaned_abstract'].fillna('')
y = df['Label']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Define four machine learning models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    ),

    'Support Vector Machine': LinearSVC(
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42
    ),

    'Complement Naive Bayes': ComplementNB()
}

# Store model performance results
results = []

# Train and evaluate each model
for name, classifier in models.items():

    # Create TF-IDF and classification pipeline
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2)
        )),
        ('classifier', classifier)
    ])

    # Train model
    pipeline.fit(X_train, y_train)

    # Make predictions
    predictions = pipeline.predict(X_test)

    # Calculate performance metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, average='weighted', zero_division=0
    )
    recall = recall_score(
        y_test, predictions, average='weighted', zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, average='weighted', zero_division=0
    )

    # Store results
    results.append({
        'Model': name,
        'Accuracy': round(accuracy * 100, 2),
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2)
    })

    # Display classification report
    print("\n", "=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

# Display model comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by='Accuracy', ascending=False
)

print("\nMODEL PERFORMANCE COMPARISON")
print(results_df.to_string(index=False))

# Identify the best performing model
best_model = results_df.iloc[0]

print("\nBEST PERFORMING MODEL:", best_model['Model'])
print("Best Accuracy:", best_model['Accuracy'], "%")

# Compare against baseline accuracy
baseline_accuracy = 89.29

print("\nBASELINE MODEL ACCURACY:", baseline_accuracy, "%")

if best_model['Accuracy'] > baseline_accuracy:
    print("The best model outperformed the baseline model.")
else:
    print("Further model optimisation is required to outperform the baseline.")



Logistic Regression
              precision    recall  f1-score   support

          ai       0.71      0.77      0.73        47
   classical       0.94      0.92      0.93       177

    accuracy                           0.88       224
   macro avg       0.82      0.84      0.83       224
weighted avg       0.89      0.88      0.89       224


Support Vector Machine
              precision    recall  f1-score   support

          ai       0.78      0.68      0.73        47
   classical       0.92      0.95      0.93       177

    accuracy                           0.89       224
   macro avg       0.85      0.82      0.83       224
weighted avg       0.89      0.89      0.89       224


Random Forest
              precision    recall  f1-score   support

          ai       1.00      0.60      0.75        47
   classical       0.90      1.00      0.95       177

    accuracy                           0.92       224
   macro avg       0.95      0.80      0.85       224
weighted avg  

In [7]:
#2.2

# Import required libraries
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

# Create Random Forest pipeline
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    ))
])

# Define hyperparameter search space
param_grid = {
    'tfidf__max_features': [1000, 1500, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'classifier__n_estimators': [200, 300, 500],
    'classifier__max_depth': [None, 20, 40],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__max_features': ['sqrt', 0.5]
}

# Apply RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train and optimise the model
random_search.fit(X_train, y_train)

# Obtain the best model
best_rf = random_search.best_estimator_

# Make predictions
tuned_predictions = best_rf.predict(X_test)

# Evaluate optimised model
tuned_accuracy = accuracy_score(y_test, tuned_predictions)

print("\nBEST HYPERPARAMETERS")
print(random_search.best_params_)

print("\nBEST CROSS-VALIDATION F1 SCORE")
print(round(random_search.best_score_, 4))

print("\nOPTIMISED MODEL ACCURACY")
print(round(tuned_accuracy * 100, 2), "%")

print("\nOPTIMISED CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    tuned_predictions,
    zero_division=0
))

# Compare original and optimised Random Forest
original_accuracy = 91.52

print("\nMODEL PERFORMANCE COMPARISON")
print("Original Random Forest:", original_accuracy, "%")
print("Optimised Random Forest:",
      round(tuned_accuracy * 100, 2), "%")

improvement = tuned_accuracy * 100 - original_accuracy

print("Performance Improvement:",
      round(improvement, 2), "percentage points")

if improvement > 0:
    print("Hyperparameter tuning improved model accuracy.")
elif improvement == 0:
    print("Model accuracy remained unchanged.")
else:
    print("Model accuracy decreased after tuning.")


Fitting 5 folds for each of 15 candidates, totalling 75 fits

BEST HYPERPARAMETERS
{'tfidf__ngram_range': (1, 2), 'tfidf__max_features': 1000, 'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 2, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20}

BEST CROSS-VALIDATION F1 SCORE
0.839

OPTIMISED MODEL ACCURACY
91.07 %

OPTIMISED CLASSIFICATION REPORT
              precision    recall  f1-score   support

          ai       0.94      0.62      0.74        47
   classical       0.91      0.99      0.95       177

    accuracy                           0.91       224
   macro avg       0.92      0.80      0.84       224
weighted avg       0.91      0.91      0.90       224


MODEL PERFORMANCE COMPARISON
Original Random Forest: 91.52 %
Optimised Random Forest: 91.07 %
Performance Improvement: -0.45 percentage points
Model accuracy decreased after tuning.


In [10]:
#2.3
import os

print(os.getcwd())

C:\Users\Mosa.Mefane


In [11]:
%%writefile app.py

import streamlit as st
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# Download NLP resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Configure application
st.set_page_config(
    page_title="Researchly Research Abstract Classifier",
    layout="wide"
)

st.title("Researchly Research Abstract Classifier")

st.write(
    "This app classifies research abstracts as either "
    "AI-related or classical research."
)

# Initialise text preprocessing
stop_words = set(stopwords.words('english')) - {'no', 'nor', 'not'}
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>|http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(words)

# Train the Random Forest model
@st.cache_resource
def train_model():

    df = pd.read_json('research_abstracts.json')

    abstract_col = next(
        col for col in df.columns if 'abstract' in col.lower()
    )

    df = df.dropna(subset=[abstract_col, 'Label']).copy()

    df['cleaned_abstract'] = df[abstract_col].apply(clean_text)

    df = df[df['cleaned_abstract'] != ''].copy()
    df = df.drop_duplicates(subset=['cleaned_abstract'])

    model = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2)
        )),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=42
        ))
    ])

    model.fit(df['cleaned_abstract'], df['Label'])

    return model

model = train_model()

# Abstract input
abstract = st.text_area(
    "Enter research abstract:",
    height=160
)

# Classification
if st.button("Classify Abstract"):

    if abstract.strip():

        cleaned = clean_text(abstract)

        if cleaned:
            prediction = model.predict([cleaned])[0]

            st.subheader("Prediction")

            if str(prediction).lower() == 'ai':
                st.success(
                    "The abstract is classified as: "
                    "Artificial Intelligence"
                )
            else:
                st.success(
                    "The abstract is classified as: Classical Research"
                )

            probabilities = model.predict_proba([cleaned])[0]

            st.write(
                "Prediction confidence:",
                f"{max(probabilities) * 100:.2f}%"
            )

            probability_df = pd.DataFrame({
                'Category': model.classes_,
                'Probability (%)': probabilities * 100
            })

            st.bar_chart(
                probability_df.set_index('Category')
            )

        else:
            st.warning("Please enter a valid research abstract.")

    else:
        st.warning("Please enter a research abstract.")

Writing app.py


In [12]:
!pip install streamlit

In [ ]:
streamlit run app.py